In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from datasets import load_dataset_builder
# dataset_name = "sms_spam"
dataset_name = "SetFit/enron_spam"
load_dataset_builder(dataset_name).info

d:\fine-tune-text-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Repo card metadata block was not found. Setting CardData to empty.


DatasetInfo(description='', citation='', homepage='', license='', features={'message_id': Value('int64'), 'text': Value('string'), 'label': Value('int64'), 'label_text': Value('string'), 'subject': Value('string'), 'message': Value('string'), 'date': Value('timestamp[s]')}, post_processed=None, supervised_keys=None, builder_name='json', dataset_name='enron_spam', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=96980930, num_examples=31716, shard_lengths=None, original_shard_lengths=None, dataset_name='enron_spam'), 'test': SplitInfo(name='test', num_bytes=6013617, num_examples=2000, shard_lengths=None, original_shard_lengths=None, dataset_name='enron_spam')}, download_checksums={'hf://datasets/SetFit/enron_spam@1916f66c89d52221ae33eb57d44498b4f3a5df22/train.jsonl': {'num_bytes': 101069043, 'checksum': None}, 'hf://datasets/SetFit/enron_spam@1916f66c89d52221ae33eb57d44498b4f3a5df22/test.jsonl': {'num_bytes': 6273613, 'checksum': None}}, download_

In [3]:
from datasets import load_dataset

dataset = load_dataset(dataset_name)
# 5,574 SMS messages (747 spam, 4,827 ham)

Repo card metadata block was not found. Setting CardData to empty.


In [4]:
dataset = dataset['train'].train_test_split(test_size=0.2)  # Split train/test

In [5]:
spam_examples = dataset['test'].filter(lambda x: x['label'] == 1)
spam_examples[0]

Filter: 100%|██████████| 6344/6344 [00:00<00:00, 69325.53 examples/s]


{'message_id': 4914,
 'text': "eshopkey + ??????????? ! ?? eshopkey\n( http : / / www . eshopkey . net )\n???????? ??? ?????????? ??? ??? ????????? ???? ???????????? ????????\n??? ???? ? ?? ???? ???????? ????? . ? ? ????????? ?? ???????? ??? ??? ?????\n??????????? ??????? ???? online ???????????? ???????????? , ??? ????? ??\n?????????? ??? ??? ????? , ???????? ??? ???????????? ??? ? ?????????? ????????\n??? internet .\n???? , ??? ??????? ??? ???????????? ' ??????????? ' , ??????????? ??? ????? ????\n???????????? ???????????? ????????????? ?????? ???? ??????? ??? ??? ??\n? ???????????? ???? ??? ???????????? .\n?? ???? ?? ????????? ' ??????????? ' ???????????? ???\n???? ??????????? ????????? ?? ?? ????\n? ????????????? :\n?????? ??? ?? 3 . 125 ?? ??? ?????? ?? 1 . 250 ???\n??? ??????????? ??? ' ??????????? ' ( ???????? ??????\n?????? ??? ???? 1 . 875 ) .\n??????????? ????? ??? ??????????????? ??? ??????\nnormal\nplan\n??? eshopkey .\n?????? ????????? ??? ???????????? ????????????\n??? 5 

# Modified Logistic Regression

## TFIDF solution

Note: 
- this gonna be tricky if we have mixed Thai and English, see 07_tokenization.ipynb for solution
- more training not help because less training performs better

In [6]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from package.models.modified_logistic_regression import ModifiedLogisticRegressionPU

# 1. Load data
dataset = load_dataset("SetFit/enron_spam")
train_data = dataset['train']

# 2. Create PU scenario (hide 75% of spam)
spam_indices = [i for i, label in enumerate(train_data['label']) if label == 1]
ham_indices = [i for i, label in enumerate(train_data['label']) if label == 0]

# Keep only 25% of spam labeled
np.random.seed(42)
labeled_spam = np.random.choice(spam_indices, size=len(spam_indices)//4, replace=False)

# Create PU labels
pu_labels = np.zeros(len(train_data))
pu_labels[labeled_spam] = 1  # Only 25% spam labeled as 1, rest are 0 (unlabeled)

# 3. Vectorize text
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(train_data['text']).toarray()
y_true = np.array(train_data['label'])  # True labels (for evaluation only)
y_pu = pu_labels  # PU labels (for training)

# 4. Train Modified Logistic Regression
model = ModifiedLogisticRegressionPU(
    lr=0.001,
    epochs=10000,
    verbose=True
)

model.fit(X, y_pu)

# 5. Predict
y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

# 6. Evaluate
print(f"\nEstimated c (labeling rate): {model.c_hat_:.4f}")
print(f"True c: {len(labeled_spam) / len(spam_indices):.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

# 7. Check probability distribution
import pandas as pd
results = pd.DataFrame({
    'true_label': y_true,
    'pu_label': y_pu,
    'predicted_proba': y_proba
})

print("\nProbability by True Label:")
print(results.groupby('true_label')['predicted_proba'].describe())


Repo card metadata block was not found. Setting CardData to empty.


Epoch 0  Loss 0.6875
Epoch 100  Loss 0.5250
Epoch 200  Loss 0.4510
Epoch 300  Loss 0.4155
Epoch 400  Loss 0.3957
Epoch 500  Loss 0.3825
Epoch 600  Loss 0.3725
Epoch 700  Loss 0.3642
Epoch 800  Loss 0.3572
Epoch 900  Loss 0.3512
Epoch 1000  Loss 0.3459
Epoch 1100  Loss 0.3413
Epoch 1200  Loss 0.3373
Epoch 1300  Loss 0.3337
Epoch 1400  Loss 0.3305
Epoch 1500  Loss 0.3276
Epoch 1600  Loss 0.3251
Epoch 1700  Loss 0.3227
Epoch 1800  Loss 0.3206
Epoch 1900  Loss 0.3187
Epoch 2000  Loss 0.3169
Epoch 2100  Loss 0.3153
Epoch 2200  Loss 0.3137
Epoch 2300  Loss 0.3123
Epoch 2400  Loss 0.3110
Epoch 2500  Loss 0.3098
Epoch 2600  Loss 0.3087
Epoch 2700  Loss 0.3076
Epoch 2800  Loss 0.3066
Epoch 2900  Loss 0.3057
Epoch 3000  Loss 0.3048
Epoch 3100  Loss 0.3040
Epoch 3200  Loss 0.3032
Epoch 3300  Loss 0.3024
Epoch 3400  Loss 0.3017
Epoch 3500  Loss 0.3010
Epoch 3600  Loss 0.3003
Epoch 3700  Loss 0.2997
Epoch 3800  Loss 0.2991
Epoch 3900  Loss 0.2985
Epoch 4000  Loss 0.2979
Epoch 4100  Loss 0.2974
Epoc

## Embeddings with PCA solution

Note: 
- it looks promising only in more training
- it still needs to find the feasible way to inspect the answers

In [7]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
from package.models.modified_logistic_regression import ModifiedLogisticRegressionPU
from sklearn.decomposition import PCA

# 1. Load BGE model
print("Loading BGE model...")
embedder = SentenceTransformer('BAAI/bge-small-en-v1.5')  # Or bge-base/large

# 2. Load data
dataset = load_dataset("SetFit/enron_spam")
train_data = dataset['train']

# 3. Create PU scenario (hide 75% of spam)
spam_indices = [i for i, label in enumerate(train_data['label']) if label == 1]
ham_indices = [i for i, label in enumerate(train_data['label']) if label == 0]

np.random.seed(42)
labeled_spam = np.random.choice(spam_indices, size=len(spam_indices)//4, replace=False)

pu_labels = np.zeros(len(train_data))
pu_labels[labeled_spam] = 1

# 4. Generate embeddings (this takes time!)
print("Generating embeddings...")
texts = train_data['text']
_X = embedder.encode(texts, show_progress_bar=True, batch_size=32)

# Reduce dimensions to match TF-IDF sparsity
pca = PCA(n_components=100)  # 384 → 100 dims
X = pca.fit_transform(_X)

y_true = np.array(train_data['label'])
y_pu = pu_labels

# 5. Train Modified Logistic Regression
print("Training PU model...")
model = ModifiedLogisticRegressionPU(
    lr=0.001,
    epochs=10000,
    verbose=True
)

model.fit(X, y_pu)

# 6. Predict
y_pred = model.predict(X)
y_proba = model.predict_proba(X)[:, 1]

# 7. Evaluate
print(f"\nEstimated c: {model.c_hat_:.4f}")
print(f"True c: {len(labeled_spam) / len(spam_indices):.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Ham', 'Spam']))

# 8. Probability analysis
import pandas as pd
results = pd.DataFrame({
    'true_label': y_true,
    'pu_label': y_pu,
    'predicted_proba': y_proba
})

print("\nProbability Distribution:")
print(results.groupby('true_label')['predicted_proba'].describe())

Loading BGE model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13262.78it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Repo card metadata block was not found. Setting CardData to empty.


Generating embeddings...


Batches: 100%|██████████| 992/992 [00:36<00:00, 27.45it/s] 


Training PU model...
Epoch 0  Loss 0.6906
Epoch 100  Loss 0.6748
Epoch 200  Loss 0.6486
Epoch 300  Loss 0.6155
Epoch 400  Loss 0.5811
Epoch 500  Loss 0.5490
Epoch 600  Loss 0.5209
Epoch 700  Loss 0.4970
Epoch 800  Loss 0.4769
Epoch 900  Loss 0.4600
Epoch 1000  Loss 0.4459
Epoch 1100  Loss 0.4341
Epoch 1200  Loss 0.4241
Epoch 1300  Loss 0.4158
Epoch 1400  Loss 0.4087
Epoch 1500  Loss 0.4027
Epoch 1600  Loss 0.3976
Epoch 1700  Loss 0.3932
Epoch 1800  Loss 0.3895
Epoch 1900  Loss 0.3863
Epoch 2000  Loss 0.3834
Epoch 2100  Loss 0.3810
Epoch 2200  Loss 0.3789
Epoch 2300  Loss 0.3770
Epoch 2400  Loss 0.3753
Epoch 2500  Loss 0.3739
Epoch 2600  Loss 0.3726
Epoch 2700  Loss 0.3714
Epoch 2800  Loss 0.3703
Epoch 2900  Loss 0.3694
Epoch 3000  Loss 0.3685
Epoch 3100  Loss 0.3677
Epoch 3200  Loss 0.3670
Epoch 3300  Loss 0.3663
Epoch 3400  Loss 0.3656
Epoch 3500  Loss 0.3650
Epoch 3600  Loss 0.3644
Epoch 3700  Loss 0.3639
Epoch 3800  Loss 0.3633
Epoch 3900  Loss 0.3628
Epoch 4000  Loss 0.3623
Epoch 4

In [8]:
# 2. Get probabilities for unlabeled data
unlabeled_proba = model.predict_proba(X)[:, 1]

# 3. Label with confidence threshold
threshold = 0.7  # Adjust based on your needs
pseudo_labels = (unlabeled_proba >= threshold).astype(int)

# 4. Filter high-confidence predictions only
high_confidence_mask = (unlabeled_proba >= threshold) | (unlabeled_proba <= 0.3)
confident_labels = pseudo_labels[high_confidence_mask]
confident_data = X[high_confidence_mask]

# 5. Combine with original labeled data for next iteration
X_combined = np.vstack([X, confident_data])
y_combined = np.hstack([y_pu, confident_labels])

# 6. Retrain (optional - self-training loop)
# model.fit(X_combined, y_combined)


In [9]:
y_pu, y_combined

(array([1., 0., 0., ..., 0., 0., 0.], shape=(31716,)),
 array([1., 0., 0., ..., 1., 1., 0.], shape=(58379,)))

In [10]:
X_combined.shape, y_combined.shape

((58379, 100), (58379,))

In [11]:
X_combined, y_combined

(array([[ 0.17110991,  0.02894376, -0.12405528, ..., -0.00319302,
         -0.01125562, -0.03438489],
        [-0.19805428,  0.03593076, -0.06973605, ..., -0.00324984,
         -0.06054849, -0.00689877],
        [ 0.27232513,  0.12765905, -0.0362748 , ...,  0.01424519,
         -0.0238388 ,  0.01443489],
        ...,
        [-0.14321321,  0.06789558, -0.09220734, ...,  0.04462387,
         -0.01280273, -0.01824706],
        [ 0.1402131 ,  0.00056288,  0.02236149, ...,  0.02737589,
         -0.0855229 ,  0.09834167],
        [-0.08763825,  0.01908018,  0.09730519, ..., -0.0007924 ,
          0.00409757,  0.03874503]], shape=(58379, 100), dtype=float32),
 array([1., 0., 0., ..., 1., 1., 0.], shape=(58379,)))

# If we wanna inspect manually

In [12]:
import pandas as pd

# Get predictions with probabilities
proba = model.predict_proba(X)[:, 1]
pred_labels = (proba >= 0.7).astype(int)

# Create DataFrame with original data
results = pd.DataFrame({
    'text': train_data["text"],  # Original text
    'predicted_label': pred_labels,
    'confidence': proba,
    'original_label': y_pu
})

In [13]:
# Filter: unlabeled that model predicts as positive
unlabeled_positive = results[(results['original_label'] == 0) & (results['predicted_label'] == 1)]

# Sort by confidence
unlabeled_positive = unlabeled_positive.sort_values('confidence', ascending=False)

# View top predictions
print(f"Found {len(unlabeled_positive)} unlabeled predicted as positive")
print(unlabeled_positive[['text', 'confidence']].head(10))

Found 15257 unlabeled predicted as positive
                                                    text  confidence
22453  findikkiran www . findikkiran . com\ntum\nalis...    0.999980
20973  findikkiran www . findikkiran . com\ntum\nalis...    0.999980
9831   mortgage interest rates are at their lowest po...    0.999975
27149  adv : win a green card and become a u . s citi...    0.999964
20052  adv : win a green card and become a u . s citi...    0.999964
25776  the smailcap journa | presents : investor aier...    0.999945
7146   kime oy vereceksiniz ? ?yi g?nler\nd?nya gazet...    0.999943
8059   kime oy vereceksiniz ? ?yi g?nler\nd?nya gazet...    0.999943
17706  calling all small stock players world stock re...    0.999930
20380  calling all small stock players world stock re...    0.999930


In [14]:
for t in unlabeled_positive.loc[:, 'text'].head(10).values.tolist():
    print(t)
    print("="*20)

findikkiran www . findikkiran . com
tum
aliskanliklarini degistiriyoruz ,
buradan asla vazgecemeyeceksin . . .
arkadas ol , sevgili bul , hatta evlen . . tum bunlar
sadece sana kalmis . . ! biz sadece araciyiz . veee tum bu hizmetleri bedava
veriyoruz . . .
bizde , gold ,
silver , altin , bronz , teneke , vip , tip uyelik
yok !
sinirli mesaj , kontor , kredi satin alma , sadece gelen mesaji cevaplayabilme
ise hic yok !
ne herhangi bi program indirmene gerek var , ne de sinir bozucu pop - up
reklamlari gormene !
rahat rahat , sinirsizca , gonlunce gez , burada
herkes esit .
evettt . . yanlis okumadin . . .
bedava !
- uye olma ,
- profil ekleme ,
- profil okuma ,
- sinirsiz mesaj gonderme , mesaj okuma . . .
kisaca verilen tum hizmetler tamamen ve de sartsiz olarak
ucretsiz !
herkes tanissin , herkes kaynassin diye : ) iyi eglenceler . .
copyright
© 2003 - 2005 findikkiran . com
alpha version 1 . 0
findikkiran www . findikkiran . com
tum
aliskanliklarini degistiriyoruz ,
buradan asla vaz

# Lesson Learned
- We can use the model to predict on the unlabeled data and get probabilities.
- We can adapt to find hard negatives (the ones with low probabilities)
- Positive or Negative Pseudo labels should be inspected or callibrate the threshold
- For the threshold, we should start small i.e. psuedo_positive >= .9 or psuedo_negative <= .1
- To make a program easy and re-usable, instead of using positive or negative, try positive and unlabeled
- y=s is the same in the sense when create training dataset